In [0]:
from pyspark.sql.functions import *
from common_scripts.dq_framework import *
import builtins

In [0]:
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<YOUR KEY>"
)


In [0]:
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"
gold_dim_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dimensions"
gold_fact_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/facts"

In [0]:
orders_df = spark.read.format("delta").load(silver_path + "orders")
order_items_df = spark.read.format("delta").load(silver_path + "order_items")
payments_df = spark.read.format("delta").load(silver_path + "payments")

In [0]:
orders_df.printSchema()
order_items_df.printSchema()
payments_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- pa

In [0]:
dim_cust = spark.read.format("delta").load(gold_dim_path + "/dim_customers")
dim_prod = spark.read.format("delta").load(gold_dim_path + "/dim_products")
dim_dates = spark.read.format("delta").load(gold_dim_path + "/dim_dates")

In [0]:
dim_cust.printSchema()
dim_prod.printSchema()
dim_dates.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_key: long (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- product_key: long (nullable = true)

root
 |-- date: date (nullable = true)
 |-- date_key: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- week: integer (nullable = true)
 |-- day: integer (nullable = true)



In [0]:
fact_df = orders_df.alias("o").join(order_items_df.alias("oi"), on="order_id", how="inner")


In [0]:
payments_df.groupBy("order_id").count() \
    .filter("count > 1") \
    .show()
    

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|0c077fbe69b84abe1...|    2|
|1826d2a2eb6ba6e3e...|    2|
|1aaeb5badaa812e15...|    2|
|2c88b5879d666444b...|    2|
|30e934394c047a409...|    2|
|35ab20ce8b706d545...|    2|
|3f4f6a378519479cb...|    2|
|54066aeaaf3ac32e7...|    2|
|5ded9a59e8920225f...|    2|
|6386b8aabb7f032b7...|    2|
|85ff97380814f8f14...|    2|
|8ca5bdac5ebe8f2d6...|    9|
|8dd9758206f8d9c23...|    2|
|ac3b0c224349e4ca9...|    3|
|b60cab8c479e1e804...|    2|
|ba1415261e29b897a...|    2|
|f44cb69655f8e4d13...|    2|
|faf1a9a55f20bf036...|    2|
|07efb6b65a21feb31...|    2|
|0fa927b252421189a...|    4|
+--------------------+-----+
only showing top 20 rows


In [0]:
payments_df.groupBy("order_id").count().orderBy("count", ascending=False).show(10)

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|fa65dad1b0e818e3c...|   29|
|ccf804e764ed5650c...|   26|
|285c2e15bebd4ac83...|   22|
|895ab968e7bb0d565...|   21|
|fedcd9f7ccdc8cba3...|   19|
|ee9ca989fc93ba09a...|   19|
|21577126c19bf11a0...|   15|
|4bfcba9e084f46c8e...|   15|
|3c58bffb70dcf45f1...|   14|
|4689b1816de42507a...|   14|
+--------------------+-----+
only showing top 10 rows


In [0]:
payments_agg = (
    payments_df
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("total_payment_value"),
        first("payment_type").alias("payment_type"),
        sum("payment_installments").alias("total_installments")
    )
)

In [0]:
fact_df = fact_df.join(payments_agg, on="order_id", how= "left")

In [0]:
fact_df.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_installments: long (nullable = true)



In [0]:
fact_df=fact_df.join(dim_cust.select("customer_id", "customer_key"), on = "customer_id", how="left")

In [0]:
fact_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- customer_key: long (nullable = true)



In [0]:
fact_df.filter("customer_key IS NULL").count()

0

In [0]:
fact_df = fact_df.join(dim_prod.select("product_id", "product_key"), on="product_id", how="left")

In [0]:
fact_df.filter("product_key IS NULL").count()

0

In [0]:
fact_df = fact_df.withColumn(
    "date_key",
    date_format(
        "order_purchase_timestamp",
        "yyyyMMdd"
    )
)

In [0]:
fact_df = (
    fact_df
    .join(
        dim_dates.select(
            "date_key"
        ),
        on="date_key",
        how="left"
    )
)

In [0]:
fact_df.printSchema()

root
 |-- date_key: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- custom

In [0]:
fact_sales = fact_df.select( "order_id",
    "order_item_id",

    "customer_key",
    "product_key",
    "date_key",

    "order_status",

    "price",
    "freight_value",

    "payment_type",
    "total_payment_value",
    "total_installments")

In [0]:
fact_sales = fact_sales.withColumn(
    "sale_key",
    monotonically_increasing_id()
)


In [0]:
#Customer key null check
run_dq_check(fact_sales,
              'fact_sales',
              'null_customer_key',
              col('customer_key').isNull(),
              'Critical'
              )

#Product key null check
run_dq_check(fact_sales,
              'fact_sales',
              'null_product_key',
              col('product_key').isNull(),
              'Critical'
              )

#Date key null check
run_dq_check(fact_sales,
              'fact_sales',
              'null_date_key',
              col('date_key').isNull(),
              'Critical'
              )

#Order status null check
run_dq_check(fact_sales,
              'fact_sales',
              'null_order_status',
              col('order_status').isNull(),
              'Warning'
              )

#Payment Type null check
run_dq_check(fact_sales,
              'fact_sales',
              'null_payment_type',
              col('payment_type').isNull(),
              'Warning'
              )

#Total payment value null check
run_dq_check(fact_sales,
              'fact_sales',
              'null_total_payment_value',
              col('total_payment_value').isNull(),
              'Warning'
              )


[CRITICAL] fact_sales | null_customer_key: 0
[CRITICAL] fact_sales | null_product_key: 0
[CRITICAL] fact_sales | null_date_key: 0
[WARNING] fact_sales | null_order_status: 0
[WARNING] fact_sales | null_payment_type: 3
[WARNING] fact_sales | null_total_payment_value: 3


3

In [0]:
failed_count = fact_sales.count() - fact_sales.select("sale_key").distinct().count()

In [0]:
run_metric_check(
    fact_sales,
    "duplicate_sale_key_check",
    failed_count,
    severity="Warning"
)

row_differnce = builtins.abs(fact_sales.count() - order_items_df.count())

run_metric_check(
    fact_sales,
    "order_items_df_row_count_check",
    row_differnce,
    severity="Critical")

[WARNING] DataFrame[order_id: string, order_item_id: string, customer_key: bigint, product_key: bigint, date_key: string, order_status: string, price: double, freight_value: double, payment_type: string, total_payment_value: double, total_installments: bigint, sale_key: bigint] | duplicate_sale_key_check: 0
[CRITICAL] DataFrame[order_id: string, order_item_id: string, customer_key: bigint, product_key: bigint, date_key: string, order_status: string, price: double, freight_value: double, payment_type: string, total_payment_value: double, total_installments: bigint, sale_key: bigint] | order_items_df_row_count_check: 0


0

In [0]:
dq_df = dq_df_from_dq_results(spark, dq_results)

In [0]:
dq_path = f"abfss://audit@{storage_account}.dfs.core.windows.net/gold_dq_logs/fact_sales"

In [0]:
dq_df.write.format('delta').mode('append').save(dq_path)

In [0]:
fact_sales.write.format('delta').mode('overwrite').save(gold_fact_path + "/fact_sales")

In [0]:
fs_df = spark.read.format('delta').load(gold_fact_path+ '/fact_sales')

fs_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- date_key: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- sale_key: long (nullable = true)

